In [1]:
# 03_fixed_effects_event_study.ipynb
# Two-way (individual + year) fixed-effects difference-in-differences for the
# effect of a health shock on catastrophic health expenditure (CHE), out-of-pocket
# spending and household income, plus a private-insurance (PHI) buffer interaction
# and a dynamic event study with a pre-trend check. Figure is grayscale, no titles,
# 600-dpi PNG and PDF. Tables go to ../results/tables/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS

PROC, FIG, TAB = "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

p = pd.read_parquet(f"{PROC}/khp_panel.parquet").sort_values(["PIDWON", "year"]).reset_index(drop=True)
p["che40f"] = p["che40"].astype(float); p["che25f"] = p["che25"].astype(float)

# ---- treatment timing: first health shock (new chronic OR new hospitalization) ----
first = p[p["shock_any"] == 1].groupby("PIDWON")["year"].min().rename("first_shock")
p = p.merge(first, on="PIDWON", how="left")
p["ever"] = p["first_shock"].notna().astype(int)
p["post_shock"] = ((p["year"] >= p["first_shock"]) & (p["ever"] == 1)).astype(int)
p["post_phi"] = p["post_shock"] * p["base_phi"]       # buffer interaction (baseline PHI)

cnt = p.groupby("PIDWON")["year"].transform("size")
est = p[cnt >= 2].set_index(["PIDWON", "year"])

def fe(y, X):
    d = est[[y] + X].dropna()
    return PanelOLS(d[y], d[X], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)

# ---- static two-way FE: main effect and PHI-buffer interaction ----
outs = [("che40f", "CHE 40%"), ("che25f", "CHE 25%"), ("ihs_oop", "asinh(OOP)"), ("log_hhinc", "log HH income")]
rows = []
for y, lab in outs:
    r1 = fe(y, ["post_shock"])
    r2 = fe(y, ["post_shock", "post_phi"])
    rows.append({"outcome": lab,
                 "shock": r1.params["post_shock"], "shock_se": r1.std_errors["post_shock"], "shock_p": r1.pvalues["post_shock"],
                 "buffer_postPHI": r2.params["post_phi"], "buffer_se": r2.std_errors["post_phi"], "buffer_p": r2.pvalues["post_phi"]})
static = pd.DataFrame(rows)
static.to_csv(f"{TAB}/t03_static_fe.csv", index=False)

# ---- dynamic event study (ref k = -1), never-treated as pure controls ----
p["k"] = p["year"] - p["first_shock"]
def kbin(k):
    return np.nan if pd.isna(k) else max(-3, min(3, int(k)))
p["kb"] = p["k"].apply(kbin)
evk = [-3, -2, 0, 1, 2, 3]
for kk in evk:
    p[f"e{kk}"] = ((p["ever"] == 1) & (p["kb"] == kk)).astype(int)
cols = [f"e{kk}" for kk in evk]
cnt2 = p.groupby("PIDWON")["year"].transform("size")
es = p[cnt2 >= 2].set_index(["PIDWON", "year"])
def esfit(y):
    d = es[[y] + cols].dropna()
    return PanelOLS(d[y], d[cols], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
ev_rows = []
coefs = {}
for y, lab in [("che40f", "CHE40"), ("ihs_oop", "asinh_OOP")]:
    r = esfit(y)
    coefs[y] = {-1: (0.0, 0.0)}
    for c in cols:
        kk = int(c[1:]); coefs[y][kk] = (r.params[c], r.std_errors[c])
        ev_rows.append({"outcome": lab, "k": kk, "coef": r.params[c], "se": r.std_errors[c], "p": r.pvalues[c]})
pd.DataFrame(ev_rows).to_csv(f"{TAB}/t03_event_study.csv", index=False)

# ---- Figure: dynamic event study (grayscale), CHE40 and asinh(OOP) ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, (y, ylab) in zip(axes, [("che40f", "Delta P(CHE 40%) vs k=-1"), ("ihs_oop", "Delta asinh(OOP) vs k=-1")]):
    ks = [-3, -2, -1, 0, 1, 2, 3]
    cf = np.array([coefs[y][k][0] for k in ks]); se = np.array([coefs[y][k][1] for k in ks])
    ax.axhline(0, color="0.5", lw=0.8); ax.axvline(-0.5, color="0.7", ls="--", lw=1)
    ax.errorbar(ks, cf, yerr=1.96 * se, fmt="o-", color="0.15", ms=6, lw=1.8, capsize=3,
                ecolor="0.45", markerfacecolor="white", markeredgewidth=1.4)
    ax.set_xlabel("Years relative to first health shock (k)"); ax.set_ylabel(ylab); ax.set_xticks(ks)
save_fig(fig, "fig05_event_study")

print("Static two-way FE (health shock effect and PHI buffer):")
print(static.round(4).to_string(index=False))
print("Event-study CHE40 pre-trend coefficients (k=-3,-2):",
      round(coefs["che40f"][-3][0], 4), round(coefs["che40f"][-2][0], 4))


Static two-way FE (health shock effect and PHI buffer):
      outcome   shock  shock_se  shock_p  buffer_postPHI  buffer_se  buffer_p
      CHE 40%  0.0313    0.0026   0.0000         -0.0349     0.0066    0.0000
      CHE 25%  0.0611    0.0038   0.0000         -0.0597     0.0096    0.0000
   asinh(OOP)  1.6924    0.0599   0.0000         -0.0266     0.1289    0.8363
log HH income -0.0387    0.0203   0.0567          0.0204     0.0400    0.6097
Event-study CHE40 pre-trend coefficients (k=-3,-2): 0.0109 0.0035
